# Person 3 — Portfolio weighting

Person 2 chooses **which stocks** to hold. Your job is to choose **how much money** to put in each selected stock. For each formation month, compare two weighting methods for both benchmarks and each $k\in\{25,50,100\}$: 12 portfolios per month.

At the end of month $t$, use the 60 months $t-59,\ldots,t$, set weights, and hold them during month $t+1$. All returns are decimals (0.01 means 1%). The functions live in `portfolio_weighting.py` so Person 4 can call them from the backtest.

## 1. Inputs and handoff issues

Run `data_extraction.ipynb` or obtain Person 1's two exported CSVs. Raw CSVs are kept locally and are not committed to Git. Person 2's `cluster_selected_stocks.csv` is included.

- `top500_data/top500_formations.csv`: month, PERMNO, capitalization, coverage flags.
- `top500_data/selected_stock_monthly_history.csv`: monthly stock returns.
- `cluster_selected_stocks.csv`: representatives for each month and k.

**Cluster membership:** Part 2 saves representatives and cluster sizes, but not member PERMNOs. We reconstruct its correlation distance, average linkage, and medoids and verify them against the saved selections. If they differ, stop and obtain memberships from Person 2; do not silently select different stocks.

**Incomplete histories:** Part 2 clusters only current top-500 stocks with 60 observed returns (382 stocks in its December 1999 example). Its clusters therefore do not cover all 500 stocks. Representative weights below redistribute the eligible benchmark mass proportionately to remain fully invested; coverage is reported explicitly. Optimization still targets the full 500-stock benchmark. To represent all 500 directly, the team needs an agreed assignment rule for excluded stocks.

**Benchmark history:** Person 1 exports holdings, but not a benchmark return series. The helper below computes each month's return from the *previous* month's top-500 holdings. Missing constituent returns invalidate the month; no zero filling or survivor reweighting is performed. Person 1 must resolve any such observations and confirm delisting handling in the CRSP extraction before the final backtest.

**Updated Person 1 contract:** retain every available monthly return for stocks ever selected, including months after they leave the top 500. Do not filter the history by same-month benchmark membership. The latest extraction change adds comments and `.copy()` to this existing selection; it does not change the selected rows.

**Export policy:** generate every holding month from January 2000 to the latest supported date. Optimization uses complete benchmark observations within each 60-month window, with at least 36 observations. Record all excluded training dates; keep missing realized benchmark returns missing.

The extended 2000–2025 sample requires a minimum of 36 complete training observations (the earlier 2020–2025 run used 48). January 2000 uses weights formed in December 1999. Its January 1995 benchmark observation is unavailable because December 1994 holdings were not exported; this date and any other incomplete training dates are recorded as exclusions.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from portfolio_weighting import (
    prepare_inputs, build_benchmark_returns, reconstruct_clusters,
    representative_weights, weight_one_month,
)

from export_monthly_weights import export_monthly_weights

DATA_DIR = Path("top500_data")  # Set this to Person 1's export folder if needed.
paths = [
    DATA_DIR / "top500_formations.csv",
    DATA_DIR / "selected_stock_monthly_history.csv",
    Path("cluster_selected_stocks.csv"),
]
missing = [str(path) for path in paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run Person 1's notebook or obtain these exports: " + ", ".join(missing)
    )
formations, history, selections = prepare_inputs(*(pd.read_csv(path) for path in paths))
print("Input rows:", len(formations), len(history), len(selections))

## 2. Construct full-universe benchmark targets

For holdings formed at the end of month $t$:

$$b^{VW}_{i,t}=\frac{M_{i,t}}{\sum_{j\in U_t} M_{j,t}},\qquad b^{EW}_{i,t}=\frac{1}{500},\qquad r^B_{t+1}=\sum_{i\in U_t}b_{i,t}r_{i,t+1}.$$

Use the historical, changing benchmark universe in each month of the optimization target. Do not backfill today's constituents to create a different benchmark.

The current export starts in January 1995, so its first computable benchmark return is February 1995. The earliest possible complete 60-month training window ends in **January 2000**, for holding in **February 2000**. With the explicit incomplete-benchmark policy, the export includes the January 2000 holding month and records the unavailable January 1995 benchmark observation as an exclusion. A full 60-observation fit for that month would require December 1994 formation holdings and resolution of other missing returns. December 2025 weights would be for January 2026, whose returns are absent from the current export.

In [ ]:
benchmarks = build_benchmark_returns(formations, history)
display(benchmarks.head())
problem_months = benchmarks.loc[benchmarks["n_missing_returns"].gt(0)]
print("Benchmark months needing Person 1's review:", len(problem_months))
display(problem_months.head(10))

## 3. Representative weights

For each cluster $C_j$, assign its representative the total benchmark mass of the stocks in that cluster, normalized over eligible stocks $E_t$:

$$w^{rep}_{j,t}=\frac{\sum_{i\in C_j}b_{i,t}}{\sum_{i\in E_t}b_{i,t}}.$$

For value weighting, this is cluster capitalization divided by total eligible capitalization. For equal weighting, it is cluster size divided by the number of eligible stocks. Equal-weighted *benchmark* does not mean equal weights on the selected representatives: a representative of 40 stocks should get more weight than one representing 4.

Example: clusters with capitalizations 60, 30, and 10 receive 60%, 30%, and 10% in the value-weighted case. If their sizes are 5, 3, and 2, they receive 50%, 30%, and 20% in the equal-weighted case.

In [ ]:
START_HOLDING = pd.Period("2000-01", freq="M")
END_HOLDING = min(
    history["month"].max(),
    formations["month"].max() + 1,
    selections["month"].max() + 1,
)
MISSING_BENCHMARK = "drop"
MIN_TRAINING_MONTHS = 36
if START_HOLDING > END_HOLDING:
    raise ValueError("The requested start is after the available data end.")

formation_month = START_HOLDING - 1
print("Holding-month range:", START_HOLDING, "through", END_HOLDING)
print("Example formation:", formation_month, "holding:", formation_month + 1)
universe, return_matrix, memberships = reconstruct_clusters(
    formation_month, formations, history, selections
)
selected_25 = selections.loc[
    selections["month"].eq(formation_month) & selections["k"].eq(25)
]
w_rep, coverage = representative_weights(
    universe, memberships.loc[memberships["k"].eq(25)], selected_25, "value_weighted"
)
print(f"Eligible share of full benchmark capitalization: {coverage:.2%}")
display(w_rep.sort_values(ascending=False).head(10))

## 4. Tracking-error optimization

Let $T$ be the number of complete benchmark months in the 60-month window ($36\leq T\leq60$ under this export policy). Let $X$ be the $T\times k$ matrix of selected stock returns and $y$ the matching historical full-index returns. Active returns are $e=Xw-y$. Stock selection still uses the full 60-month history; only the fitting observations are reduced.

The default solves:

$$\min_w\;\frac{1}{T-1}\sum_{s=1}^{T}(e_s-\bar e)^2
\quad\text{subject to}\quad \sum_{j=1}^{k}w_j=1,\quad 0\leq w_j\leq u.$$

This minimizes **tracking-error variance**; its square root times $\sqrt{12}$ is annualized tracking error. Set `objective="mse"` to minimize $\frac{1}{T}\sum e_s^2$ instead. MSE penalizes average tracking difference as well as variability. Neither objective uses next month's returns. The solver uses [SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html), with an analytic gradient and explicit success/constraint checks.

- `max_weight=1.0`: no additional concentration cap. For a 10% cap use `0.10`. Feasibility requires $ku\geq1$. The cap applies to optimized portfolios; the representative baseline keeps the exact cluster mass rule.
- `ridge=0.0`: pure objective. Positive ridge adds $\lambda\|w-w^{rep}\|_2^2$, which shrinks weights toward the representative baseline. Choose any tuning value using only past information.
- At $k=100$ with at most 60 observations, the covariance is singular and the unregularized optimum may be nonunique. The constrained solver does not invert it, but fitted performance can overstate future performance. Ridge can stabilize the fit.
- Some optimal weights may be zero: k is the selected candidate count, not a guarantee of k positive positions. No sector or beta constraints are imposed by default.

In [ ]:
OBJECTIVE = "tracking_error"
MAX_WEIGHT = 1.0
RIDGE = 0.0

weights_example, diagnostics_example = weight_one_month(
    formation_month, formations, history, selections, benchmarks,
    objective=OBJECTIVE, max_weight=MAX_WEIGHT, ridge=RIDGE,
    missing_benchmark=MISSING_BENCHMARK, min_training_months=MIN_TRAINING_MONTHS,
)
display(diagnostics_example[[
    "k", "benchmark", "method", "n_positive_weights", "max_weight",
    "n_training_months", "excluded_training_months",
    "benchmark_mass_covered", "in_sample_te_annualized",
    "in_sample_mean_active_monthly", "in_sample_rmse_monthly",
]])

## 5. Export weights for Person 4

The notebook and `export_monthly_weights.py` call the same export function. Every requested holding month produces 12 portfolios. Incomplete benchmark observations are excluded from the fitting window and listed in the diagnostics, along with the actual training count. A window with fewer than 36 usable observations stops execution; it does not silently skip a holding month. Solver failures and mismatched cluster assignments also stop execution.

The three files are `monthly_weights_START_to_END.csv`, `monthly_weight_diagnostics_START_to_END.csv`, and `benchmark_returns_START_to_END.csv` in `part3_output/`. The benchmark audit includes the earlier training period as well as the holding period. These names match the command-line exports.

The diagnostics are **in-sample**, not evidence of out-of-sample tracking quality. Missing holding-month benchmark returns still need resolution before the final backtest.

In [ ]:
portfolio_weights, weight_diagnostics, benchmarks = export_monthly_weights(
    formations, history, selections,
    output_dir="part3_output", start=START_HOLDING, end=END_HOLDING,
    missing_benchmark=MISSING_BENCHMARK, min_training_months=MIN_TRAINING_MONTHS,
    objective=OBJECTIVE, max_weight=MAX_WEIGHT, ridge=RIDGE,
)
display(weight_diagnostics[[
    "holding_month", "k", "benchmark", "method", "n_training_months",
    "n_excluded_training_months", "excluded_training_months",
]].head(12))

## 6. What Person 4 does next

`monthly_weights_START_to_END.csv` contains `formation_month`, `holding_month`, `k`, `benchmark`, `method`, `permno`, `weight`. Join each positive-weight position to `history` using **holding_month and permno**, then calculate $r^P_{t+1}=\sum_jw_{j,t}r_{j,t+1}$. Missing held-stock returns need an explicit data/delisting policy; never silently skip them. Join `benchmark_returns_START_to_END.csv` on holding month and compare to the matching benchmark column.

Report out-of-sample annualized tracking error, return correlation, cumulative returns, and average tracking difference for each of the 12 strategies. The optimizer's in-sample advantage need not survive out of sample.

For turnover at rebalance $t$, first drift the old weights using the just-realized returns:

$$\widetilde w_{i,t}=\frac{w_{i,t-1}(1+r_{i,t})}{1+r^P_t},\qquad
\mathrm{turnover}_t=\frac12\sum_i|w_{i,t}-\widetilde w_{i,t}|.$$

Align PERMNOs over the union of old and new holdings so liquidations count. Decide how to report initial investment, delisting proceeds, and any omitted months. Do not treat separated valid months as adjacent rebalances. Transaction costs and an optional turnover penalty are extensions; this baseline does not optimize turnover.

**Your presentation:** explain the cluster mass rule, show the constrained optimization equation, report coverage and concentration, then let Person 4's future-month results determine which weighting method works best.